In [1]:
from Bio.UniProt.GOA import GAF20FIELDS
from collections import defaultdict
from tqdm.auto import tqdm
import gzip
import itertools
import pandas as pd
import os

from hogprop.OBOParser import OBO

# generate the results table
def gen():
    methods = ['InterProScan', 'fantasia', 'eggnog-mapper']
    method_header = {'InterProScan': 'interpro_scan',
                     'fantasia': 'fantasia',
                     'eggnog-mapper': 'eggnog_mapper'}
    for (hog_id, z) in tqdm(annotated.items()):
        for t in sorted(set(itertools.chain.from_iterable(z.values()))):
            r = {'hog_id': hog_id,
                 'go_id': str(go[t])}
            for m in methods:
                r[method_header[m]] = (t in z[m])

            yield r

In [2]:
go = OBO('../data/geneontology/go.obo', store_as_int=True)

In [3]:
res_fn = 'hog_function_source.tsv.gz'
if os.path.isfile(res_fn):
    res_df = pd.read_csv(res_fn, sep='\t')
else:
    # Load root hog definition
    df = pd.read_csv('./result/RootHOGs.tsv', sep='\t')
    df = df[['RootHOG', 'Protein']].rename(columns={'RootHOG': 'hog_id', 'Protein': 'DB_Object_ID'})
    
    # Load all predictions (filtered to QuickGO Viridiplantae)
    pbar = tqdm()
    annotated = defaultdict(lambda: defaultdict(set))
    for zdf in pd.read_csv('../data/functional_predictions/all_predictions_filtered_viridiplantae.gaf.gz', sep='\t', names=GAF20FIELDS, comment='!', chunksize=int(1e6)):
        pbar.update(len(zdf))
        zdf = pd.merge(zdf, df, on='DB_Object_ID', how='inner')
        zdf = zdf[['hog_id', 'GO_ID', 'DB']]
        zdf = zdf.drop_duplicates()
    
        for (method, zzdf) in zdf.groupby('DB'):
            for (hog_id, zzzdf) in zzdf.groupby('hog_id'):
                ts = set(zzzdf.GO_ID)
                all_terms = set()
                for t in ts:
                    all_terms.update(go.parents(t, include_self=True))
                annotated[hog_id][method].update(all_terms)
    pbar.close()

    res_df = pd.DataFrame(gen())

    with gzip.open(res_fn, 'wt') as fp:
        res_df.to_csv(fp, sep='\t', index=False)

In [4]:
res_df.hog_id.value_counts().mean()

122.63713379435349